# 02: PCA Exploration

Two principal component analyses on the filtered player-seasons:

- **Part A: all numeric columns** of `players_full` (about 880 before cleaning). This shows what drives variation in the raw data when nothing is filtered by hand.
- **Part B: the 32 style features** from `player_prep.build_style_features`. This shows the structure AA will work with.
- **Part C: comparison.** How strongly each PCA's leading components track playing time and efficiency, i.e. quality rather than style.

The sample filters and features come from `player_prep.py`, so this notebook always matches notebook 01.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# paths.py and player_prep.py live in the repo root, one level up from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

import paths
import player_prep as prep

print(sys.executable)

## 2. Load the filtered sample and the style features

In [ ]:
pf = prep.load_filtered_players()
feat, ids = prep.build_style_features(pf)
print(f"\nStyle features: {feat.shape[0]:,} rows x {feat.shape[1]} features")

# Column-to-section lookup, used to interpret loadings
data_dict = paths.load(paths.DATA_DICT)
pf_dict = data_dict[data_dict["table"].str.contains("players_full")]
section_of = dict(zip(pf_dict["column"], pf_dict["section"]))

## 3. Helper functions

`WITHIN_SEASON = True` standardizes each feature within its season (mean 0, sd 1 per season), matching the modeling plan. This removes league-wide trends across seasons, such as rising three-point rates, so that components describe differences between players rather than differences between years. Set it to `False` to standardize across all seasons at once.

In [ ]:
WITHIN_SEASON = True
REFERENCE_COLS = ["Min", "Usage %", "TS %", "eFG %", "Min Share %"]  # volume and quality proxies


def standardize(X, seasons, within_season=WITHIN_SEASON, min_std=1e-8):
  """Z-score each column, either within season or across the whole sample.

  Columns that are (nearly) constant within a season get a z-score of 0 there,
  rather than an infinite value from dividing by a standard deviation of ~0.
  """
  X = X.astype("float64")
  if within_season:
    grouped = X.groupby(seasons)
    mean = grouped.transform("mean")
    std = grouped.transform("std")
  else:
    mean = X.mean()
    std = X.std()
  std = std.where(std > min_std)  # near-zero std becomes missing
  Z = (X - mean) / std
  return Z.replace([np.inf, -np.inf], np.nan).fillna(0)


def fit_pca(Z, n_components=30):
  """Fit PCA and return the model and a DataFrame of component scores."""
  k = min(n_components, Z.shape[1])
  pca = PCA(n_components=k, random_state=0)
  scores = pca.fit_transform(Z.values)
  scores = pd.DataFrame(scores, index=Z.index, columns=[f"PC{i + 1}" for i in range(k)])
  return pca, scores


def scree(pca, title):
  """Bar chart of explained variance per component, with cumulative line."""
  var = pca.explained_variance_ratio_
  cum = np.cumsum(var)
  x = np.arange(1, len(var) + 1)
  fig, ax = plt.subplots(figsize=(9, 4))
  ax.bar(x, 100 * var, color="steelblue", label="Per component")
  ax2 = ax.twinx()
  ax2.plot(x, 100 * cum, color="black", marker="o", markersize=3, label="Cumulative")
  ax2.set_ylim(0, 100)
  ax.set_xlabel("Component")
  ax.set_ylabel("% variance explained")
  ax2.set_ylabel("Cumulative %")
  ax.set_title(title)
  plt.tight_layout()
  plt.show()
  for target in [0.5, 0.8, 0.9]:
    n = int(np.searchsorted(cum, target) + 1)
    if n <= len(cum):
      print(f"Components for {target:.0%} of variance: {n}")
    else:
      print(f"Components for {target:.0%} of variance: more than {len(cum)}")


def top_loadings(pca, columns, n_pcs=5, n_top=10, with_section=False):
  """Largest loadings (by absolute value) for each of the first n_pcs components."""
  load = pd.DataFrame(pca.components_.T, index=columns,
                      columns=[f"PC{i + 1}" for i in range(pca.n_components_)])
  for pc in load.columns[:n_pcs]:
    top = load[pc].reindex(load[pc].abs().sort_values(ascending=False).index).head(n_top)
    out = top.round(3).to_frame("loading")
    if with_section:
      out["section"] = [section_of.get(c, "(derived)") for c in out.index]
    var = pca.explained_variance_ratio_[int(pc[2:]) - 1]
    print(f"\n{pc} ({var:.1%} of variance)")
    display(out)
  return load


def section_shares(load, n_pcs=5):
  """Share of each component's squared loadings that comes from each data dictionary section."""
  sq = load.iloc[:, :n_pcs] ** 2
  sq["section"] = [section_of.get(c, "(derived)") for c in sq.index]
  shares = sq.groupby("section").sum()
  return (100 * shares / shares.sum()).round(1).sort_values("PC1", ascending=False)


def reference_corr(scores, n_pcs=5):
  """Correlation of component scores with volume and quality proxies."""
  ref = pf[[c for c in REFERENCE_COLS if c in pf.columns]]
  return pd.DataFrame({
    pc: ref.corrwith(scores[pc]) for pc in scores.columns[:n_pcs]
  }).round(2)

## Part A: PCA on all numeric columns

### A1. Build the matrix

In [ ]:
# All numeric columns, excluding identifiers and our helper columns
num = pf.select_dtypes("number")
num = num.drop(columns=[c for c in num.columns if "id" in c.lower().split()] + prep.HELPER_COLS, errors="ignore")
n_numeric = num.shape[1]

# Ratio columns can be infinite when the denominator is 0; treat those as missing
n_inf = np.isinf(num).sum()
print("Columns with infinite values:")
print(n_inf[n_inf > 0].sort_values(ascending=False).to_string(), "\n")
num = num.replace([np.inf, -np.inf], np.nan)

# Drop mostly-missing columns, fill remaining gaps with medians, drop constant columns
MAX_MISSING = 0.2
num = num.loc[:, num.isna().mean() <= MAX_MISSING]
n_after_missing = num.shape[1]
num = num.fillna(num.median())
num = num.loc[:, num.std() > 0]

# Drop columns that are constant within every season (metadata, not player stats)
within_std = num.groupby(pf["Season"]).std()
season_constant = within_std.columns[(within_std.fillna(0) < 1e-8).all()]
print(f"Constant within every season: {list(season_constant)}")
num = num.drop(columns=season_constant)

# Review numeric columns from the Info & Status section, which may also be metadata
info_cols = [c for c in num.columns if section_of.get(c) == "Info & Status"]
print(f"Info & Status columns still included: {info_cols}")

print(f"Final columns for PCA: {num.shape[1]}")

print(f"Numeric columns:                 {n_numeric}")
print(f"After dropping >{MAX_MISSING:.0%} missing:       {n_after_missing}")
print(f"After dropping constant columns: {num.shape[1]}")

### A2. Fit and explained variance

In [ ]:
Z_all = standardize(num, pf["Season"])
pca_all, scores_all = fit_pca(Z_all, n_components=30)
scree(pca_all, "PCA on all numeric columns")

### A3. What each component measures

Signs are arbitrary: a component and its negative are equivalent, so read loadings in terms of which columns move together and which move opposite.

In [ ]:
load_all = top_loadings(pca_all, num.columns, n_pcs=5, n_top=10, with_section=True)

Share of each component's squared loadings by data dictionary section. A component dominated by Counting Totals or Per Game sections is mostly measuring volume.

In [ ]:
section_shares(load_all, n_pcs=5)

### A4. Do the components track playing time and efficiency?

In [ ]:
reference_corr(scores_all, n_pcs=5)

## Part B: PCA on the 32 style features

### B1. Missing values

Missing values here are informative (for example, `3PM Assisted` is missing for players with no made threes). For this exploration only, they are filled with the feature median; the modeling step will handle them more carefully.

In [ ]:
missing = feat.isna().sum()
print("Missing values filled with medians (exploration only):")
print(missing[missing > 0].to_string())

feat_filled = feat.fillna(feat.median())

### B2. Fit and explained variance

In [ ]:
Z_style = standardize(feat_filled, pf["Season"])
pca_style, scores_style = fit_pca(Z_style, n_components=feat.shape[1])
scree(pca_style, "PCA on the 32 style features")

### B3. What each component measures

In [ ]:
load_style = top_loadings(pca_style, feat.columns, n_pcs=5, n_top=8)

### B4. Do the components track playing time and efficiency?

In [ ]:
reference_corr(scores_style, n_pcs=5)

### B5. Players on the first two style components

In [ ]:
three_share = feat_filled["Corner 3"] + feat_filled["Wing 3"] + feat_filled["Top of Key 3"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
panels = [(feat_filled["Rim"], "Rim share of FGA (%)"), (three_share, "Three-point share of FGA (%)")]
for ax, (color, label) in zip(axes, panels):
  sc = ax.scatter(scores_style["PC1"], scores_style["PC2"], c=color, s=4, alpha=0.4, cmap="viridis")
  fig.colorbar(sc, ax=ax, label=label)
  ax.set_xlabel("Style PC1")
  ax.set_ylabel("Style PC2")
  ax.set_title(f"Colored by {label.split(' (')[0].lower()}")
plt.tight_layout()
plt.show()

## Part C: Comparison

Largest absolute correlation between each reference column and the first three components of each PCA. If the all-features PCA tracks minutes and efficiency much more strongly than the style PCA, that supports building archetypes from style features only.

In [ ]:
def max_abs_corr(scores, n_pcs=3):
  return reference_corr(scores, n_pcs=n_pcs).abs().max(axis=1)

comparison = pd.DataFrame({
  "All features (max |r|, PC1-3)": max_abs_corr(scores_all),
  "Style features (max |r|, PC1-3)": max_abs_corr(scores_style),
})
comparison

In [ ]:
var_compare = pd.DataFrame({
  "All features": 100 * pca_all.explained_variance_ratio_[:5],
  "Style features": 100 * pca_style.explained_variance_ratio_[:5],
}, index=[f"PC{i}" for i in range(1, 6)]).round(1)
print("% of variance explained by each of the first five components")
var_compare

## Notes

**Settings used:** `WITHIN_SEASON = True`; all-features `MAX_MISSING = 0.2`; missing style values filled with medians (exploration only). Dropped from the all-features matrix as metadata: Next/Prior Competition Id, Min Threshold, League Pace, Jersey Num.

**Part A: all numeric columns (833 columns)**
- 4 components explain 50% of variance; 30 components explain only about 76%. A few large components followed by a long, flat tail indicates heavy redundancy (many near-duplicate columns).
- PC1 (22.4%): scoring volume. Top loadings are all season totals (FGM, points, shot attempts, points created), loading nearly equally.
- PC2 (17.7%): interior vs. perimeter (average shot distance, three-point rate vs. rim share, layups/dunks).
- PC3 (6.6%): assisted vs. self-created shots.
- PC4 (3.7%): midrange scorer vs. playmaker.
- PC5 (2.7%): team totals and games played (context, not player style).
- Section shares mainly reflect how many columns each section has (Shot Zones has 283), so loadings are more informative than the section table.

**Part B: 32 style features**
- 5 components for 50%, 13 for 80%, 19 for 90%. The last 3-4 components explain almost nothing because location shares and passing shares each sum to about 100% (compositional constraint).
- PC1 (27.5%): interior vs. perimeter. PC2 (10.7%): on-ball creator vs. off-ball spot-up. PC3 (5.8%): midrange shooter. PC4 (5.3%): passing destination (rim vs. above-the-break threes). PC5 (4.8%): lob-threat big.
- Players on PC1 vs. PC2 form a rough triangle (three-point specialists, rim bigs, on-ball creators), the geometry AA is designed for.
- Correlation of PC1 with eFG% (0.32) reflects shot mix (rim shots convert more often), not quality.

**Part C: comparison (max |r| with PC1-3)**
- Minutes: 0.80 (all) vs. 0.28 (style); Min Share: 0.78 vs. 0.36; TS%: 0.52 vs. 0.28; eFG%: 0.58 vs. 0.32; Usage: 0.62 vs. 0.51.
- The raw data's dominant structure is volume and playing time; the style features largely remove it. This supports building archetypes from style features only.

**Implications for AA**
1. Handle the compositional constraint: drop one location share and one passing share, or use a log-ratio transform.
2. `3PM Assisted` is missing for 2,609 player-seasons (no made threes); use a neutral fill plus an indicator rather than the median.
3. Add `Usage %` to the with/without check alongside `TO %`, `Self OReb Share`, and `Blocked Per FGA`.
4. No sharp elbow after PC2: try K from about 3 to 10 and compare reconstruction error, stability, and interpretability.